# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** A page is worth a CTR-fix review if it's genuinely visible
(enough impressions to matter) but earns far fewer clicks than other pages sitting at the same
search position typically do. Two signals checked below before coding anything.


In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df.shape)


FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/content_refresh_anonymized.csv'

**Signal A — staleness (`age_tier`) vs. decline rate** (the signal behind FlyRank's refresh
flag: "old content is more likely to be declining").


In [ ]:
signal_a = df.groupby("age_tier")["is_declining"] if "is_declining" in df.columns else None
df["is_declining"] = (df["trend_direction"] == "down").astype(int)  # for THIS CHECK ONLY -- never used as a rule input

bucket_a = df.groupby("age_tier")["is_declining"].agg(["mean", "count"]).sort_index()
print(bucket_a)


**Verdict: OPPOSITE.** Decline rate actually *drops* as content gets older
(31-90d: 67% declining, n=492 → 365+d: 43% declining, n=6,360) — the reverse of "stale content
decays more." A clean, explained negative — this is exactly why staleness gets left OUT of the
rule below, instead of assumed in.

**Signal B — CTR vs. position (`position_tier`)** (the signal behind FlyRank's CTR-fix logic:
"better position should mean better CTR").


In [ ]:
bucket_b = df.groupby("position_tier")["ctr"].agg(["mean", "count"]).sort_values("mean")
print(bucket_b)


**Verdict: CONFIRMED.** CTR rises cleanly and monotonically with position — `deep`
(0.15%, n=1,319) → `page_3_5` (0.22%, n=7,242) → `striking` (0.32%, n=7,304) → `page_1` (0.65%,
n=11,814) → `top_3` (1.48%, n=2,321). This one holds up and becomes the backbone of the rule.

**Reason code:** `ctr_below_position_expectation` — one code, attached to every flagged row.
**Action label:** `review_ctr_fix`.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = (CTR is under half its position tier's typical rate) × (visible: ≥500 impressions) ×
impressions — readable on purpose, no fitted weights. Staleness is deliberately absent, per
Signal A above.


In [ ]:
import os

expected_ctr_for_tier = df.groupby("position_tier")["ctr"].transform("mean")  # tier average, frozen, not a fitted weight
underperforms_ctr = (df["ctr"] < expected_ctr_for_tier * 0.5).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["score"] = underperforms_ctr * visible * df["impressions_90d"]
df["reason_code"] = "ctr_below_position_expectation"
df["action"] = "review_ctr_fix"

print("rows scored > 0:", (df["score"] > 0).sum(), "of", len(df))

os.makedirs("../outputs", exist_ok=True)
out_cols = ["content_id", "position_tier", "avg_position", "ctr", "impressions_90d", "clicks_90d", "score", "reason_code", "action"]
ranked = df.sort_values("score", ascending=False)[out_cols]
ranked.to_csv("../outputs/baseline_action_score.csv", index=False)
print("written: work/outputs/baseline_action_score.csv")
ranked.head(10)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Required top-10 shown here (the skeleton title says top-20 — that deeper version is optional,
per the card; ten is the required bar).


In [ ]:
review_cols = ["content_id", "content_type", "position_tier", "avg_position", "ctr", "impressions_90d", "clicks_90d", "main_intent", "score"]
top10 = df.sort_values("score", ascending=False)[review_cols].head(10).reset_index(drop=True)
top10


**Hand review, one line each — action / why / what would make it wrong:**

1. **`content_5fe46e04994d`** — `review_ctr_fix`. `page_1` (pos 4.2) but CTR 0.14% vs. tier's
   ~0.65% typical, on 517,715 impressions — huge gap at huge volume. *Wrong if:* the query is
   informational and a featured snippet is eating clicks the title/meta can't win back.
2. **`content_aaef01a50def`** — same pattern, `page_1`, CTR 0.25% on 517,109 impressions.
   *Wrong if:* same snippet-competition risk as #1.
3. **`content_8c19996aa890`** — `top_3` (pos 2.5), CTR only 0.15% vs. tier's ~1.48% typical —
   the single biggest relative gap in the list. *Wrong if:* the ranking is volatile
   day-to-day and this snapshot caught an unlucky dip, not a persistent problem.
4. **`content_2cb567c3c89b`** — `page_3_5` but `avg_position` 22.2, near the tier's worst edge.
   *Wrong if:* the tier-level benchmark is too coarse — position 22 isn't really comparable to
   position 4 inside the same tier.
5. **`content_4c36c775b818`** — `top_3` (pos 2.3), CTR 0.41% vs. ~1.48% typical. *Wrong if:*
   same snippet/answer-box risk as #1 and #3.
6. **`content_1a9e894be2e2`** — `page_1`, CTR 0.23%, `main_intent = transactional`.
   *Wrong if:* actually the opposite — transactional queries are usually higher-value to fix,
   so this might be under-prioritized relative to the purely score-ranked position, not wrong to flag.
7. **`content_db5989a78dd3`** — `page_1`, CTR 0.21%, `main_intent = commercial`. *Wrong if:*
   the title/meta already matches intent well and the real cause is a stronger competing result,
   which a CTR-fix review can't repair.
8. **`content_44e481c8f55b`** — `top_3` (pos 1.4, the best rank in the whole list), CTR 0.65% —
   still under the tier's 1.48% typical, but the smallest relative gap here. *Wrong if:* this
   one shouldn't have scored as high as it did relative to worse-gap rows — worth sanity-checking
   the raw score math for edge cases near the tier average.
9. **`content_cb112fce36be`** — `page_1`, CTR 0.16%, `main_intent = transactional`. *Wrong if:*
   same as #6 — intent-based re-weighting might change its priority, not its validity.
10. **`content_36ff89c8214e`** — `page_1` but `avg_position` 7.3 (worst edge of that tier), CTR
    0.05% — the lowest CTR in the whole top 10. *Wrong if:* same tier-boundary issue as #4 —
    position 7.3 behaves more like `page_3_5` than `page_1`.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest picks: #4 and #10.** Both sit at the ragged edge of their `position_tier` bucket
(positions 22.2 and 7.3 respectively) rather than deep inside it, so the tier-level average CTR
they're compared against doesn't really represent their true expected value — a continuous
position-based benchmark would likely score them differently. This is a real limitation of using
tier averages instead of a smooth position curve, not a bug.

**Leakage check:** the score uses only `ctr`, `position_tier`, and `impressions_90d` — all
same-window (trailing-90-day) fields already present in the current snapshot. `trend_pct`,
`trend_direction`, and `is_declining` (used only above, for the Signal A check, never as a rule
input) are absent from the score entirely.


In [2]:
score_inputs = ["ctr", "position_tier", "impressions_90d"]
excluded_from_rule = ["trend_pct", "trend_direction", "is_declining"]

print("columns feeding the score:", score_inputs)
print("confirmed absent from the score:", excluded_from_rule)
assert not any(c in score_inputs for c in excluded_from_rule), "leakage check failed"
print("leakage check passed: no future-window or label-derived column feeds the rule")


columns feeding the score: ['ctr', 'position_tier', 'impressions_90d']
confirmed absent from the score: ['trend_pct', 'trend_direction', 'is_declining']
leakage check passed: no future-window or label-derived column feeds the rule
